In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

TARGET_COL = 'accepted' 
CUSTOMER_ID_COL = 'customer_id'

FILE_TRAIN = 'invest_train.csv'
FILE_TEST_PUBLIC = 'invest_test_private.csv'
FILE_SAMPLE_SUBMISSION = 'sample_submission.csv' 

print("Загрузка данных...")
df_train = pd.read_csv(FILE_TRAIN)
df_test = pd.read_csv(FILE_TEST_PUBLIC)
df_sample = pd.read_csv(FILE_SAMPLE_SUBMISSION)

df_test_processed = df_test.copy()
df_test_processed.loc[:, TARGET_COL] = np.nan 

df_full = pd.concat([df_train, df_test_processed], ignore_index=True)

test_ids = df_test[CUSTOMER_ID_COL].values

print(f"Обучающая выборка: {len(df_train)} строк. Тестовая выборка: {len(df_test)} строк.")
print(f"Полный набор для обработки (df_full): {len(df_full)} строк.")
print("Первые 5 строк объединенного набора:")
display(df_full.head())

Загрузка данных...
Обучающая выборка: 6000 строк. Тестовая выборка: 1000 строк.
Полный набор для обработки (df_full): 7000 строк.
Первые 5 строк объединенного набора:


,customer_id,age,balance,risk_profile,marketing_channel,offer_amount,previous_investments,responded_before,membership_tier,accepted
0,0,75,199122.20,low,phone,79471,0,0,platinum,0.0
1,1,72,182826.57,high,in_branch,59413,1,1,gold,1.0
2,2,82,119785.23,high,sms,33092,1,1,gold,1.0
3,3,23,320109.79,high,sms,34806,1,0,standard,0.0
4,4,68,166134.85,low,email,66491,0,0,standard,0.0


In [2]:
print("Кодирование и усиленная инженерия признаков...")

df_full['responded_before'] = df_full['responded_before'].astype(int)
df_full['previous_investments'] = df_full['previous_investments'].astype(int)

df_full.loc[:, 'is_high_propensity'] = (
    (df_full['responded_before'] == 1) & 
    (df_full['previous_investments'] == 1)
).astype(int)

df_full.loc[:, 'balance_to_offer_ratio'] = (
    df_full['balance'] / (df_full['offer_amount'] + 1e-6)
)
df_full.replace([np.inf, -np.inf], np.nan, inplace=True)

df_full.loc[:, 'log_balance'] = np.log1p(df_full['balance'])
df_full.loc[:, 'log_offer_amount'] = np.log1p(df_full['offer_amount'])
df_full.loc[:, 'log_age'] = np.log1p(df_full['age'])


AGE_THRESHOLD = 55


df_full.loc[:, 'is_elite_investor'] = (
    (df_full['membership_tier'] == 'platinum') & 
    (df_full['risk_profile'] == 'low') &
    (df_full['age'] < AGE_THRESHOLD)
).astype(int)
print(f"Добавлен 'is_elite_investor' (Platinum, Low Risk, Age < {AGE_THRESHOLD})")

df_full.loc[:, 'is_conservative'] = (
    (df_full['age'] >= AGE_THRESHOLD) &
    (df_full['risk_profile'].isin(['medium', 'high']))
).astype(int)


df_full.loc[:, 'ratio_x_high_propensity'] = (
    df_full['is_high_propensity'] * df_full['balance_to_offer_ratio']
)

df_full.loc[:, 'risk_x_channel'] = (
    df_full['risk_profile'] + '_' + df_full['marketing_channel']
)

categorical_cols = ['risk_profile', 'marketing_channel', 'membership_tier', 'risk_x_channel']

df_full_encoded = pd.get_dummies(
    df_full, 
    columns=categorical_cols, 
    dummy_na=False, 
    drop_first=False
)

columns_to_drop = [
    'customer_id',
    'risk_profile', 
    'marketing_channel', 
    'membership_tier', 
    'risk_x_channel'
]

df_full_encoded = df_full_encoded.drop(
    columns=[col for col in columns_to_drop if col in df_full_encoded.columns],
    errors='ignore'
)

print("Признаки закодированы.")
display(df_full_encoded.head(2))

Кодирование и усиленная инженерия признаков...
Добавлен 'is_elite_investor' (Platinum, Low Risk, Age < 55)
Признаки закодированы.


,age,balance,offer_amount,previous_investments,responded_before,accepted,is_high_propensity,balance_to_offer_ratio,log_balance,log_offer_amount,...,risk_x_channel_high_phone,risk_x_channel_high_sms,risk_x_channel_low_email,risk_x_channel_low_in_branch,risk_x_channel_low_phone,risk_x_channel_low_sms,risk_x_channel_medium_email,risk_x_channel_medium_in_branch,risk_x_channel_medium_phone,risk_x_channel_medium_sms
0,75,199122.20,79471,0,0,0.0,0,2.505596,12.201679,11.283160,...,False,False,False,False,True,False,False,False,False,False
1,72,182826.57,59413,1,1,1.0,1,3.077215,12.116299,10.992285,...,False,False,False,False,False,False,False,False,False,False


In [3]:
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
import lightgbm as lgbm
from sklearn.metrics import f1_score
import joblib

TARGET_COL = 'accepted'
RANDOM_SEED = 42
N_FOLDS = 5
N_ESTIMATORS = 3000
LEARNING_RATE = 0.01

def find_optimal_threshold(y_true, y_proba):
    """Находит порог, максимизирующий F1-Score."""
    thresholds = np.linspace(0.3, 0.7, 50) 
    best_f1 = 0
    best_threshold = 0.5
    
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        f1 = f1_score(y_true, y_pred)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = t
            
    return best_threshold, best_f1

print("Разделение данных...")
df_train_ready = df_full_encoded[df_full_encoded[TARGET_COL].notna()].copy()
df_test_ready = df_full_encoded[df_full_encoded[TARGET_COL].isna()].drop(columns=[TARGET_COL]).copy()

Y = df_train_ready[TARGET_COL].astype(int)
X = df_train_ready.drop(columns=[TARGET_COL])

count_0 = (Y == 0).sum()
count_1 = (Y == 1).sum()
scale_pos_weight = count_0 / count_1 

print(f"Вес положительного класса (scale_pos_weight): {scale_pos_weight:.2f}")

lgbm_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'n_estimators': N_ESTIMATORS,
    'learning_rate': LEARNING_RATE,
    'random_state': RANDOM_SEED, 
    'n_jobs': 4,
    'verbose': -1,
    'scale_pos_weight': scale_pos_weight, 
    'num_leaves': 63,        
    'max_depth': 10,         
    'min_child_samples': 10  
}

kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
cv_scores = []
models = []
optimal_thresholds = []

print(f"\nНачало {N_FOLDS}-кратной кросс-валидации...")
for fold, (train_index, val_index) in enumerate(kf.split(X, Y)):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    Y_train, Y_val = Y.iloc[train_index], Y.iloc[val_index]

    model = LGBMClassifier(**lgbm_params)

    model.fit(X_train, Y_train,
              eval_set=[(X_val, Y_val)],
              eval_metric='binary_logloss',
              callbacks=[lgbm.early_stopping(100, verbose=False)])

    y_proba_val = model.predict_proba(X_val)[:, 1]

    best_t, best_f1 = find_optimal_threshold(Y_val, y_proba_val)
    
    cv_scores.append(best_f1)
    models.append(model)
    optimal_thresholds.append(best_t)
    
    print(f"--- ФОЛД {fold + 1}/{N_FOLDS} --- Оптимальный порог: {best_t:.3f}, F1-Score: {best_f1:.4f}")

mean_f1 = np.mean(cv_scores)
std_f1 = np.std(cv_scores)
mean_t = np.mean(optimal_thresholds)

print("-------------------------------------------")
print(f"Средний F1-Score по {N_FOLDS} фолдам: {mean_f1:.4f} (±{std_f1:.4f})")
print(f"Средний Оптимальный Порог: {mean_t:.3f}")
print("-------------------------------------------")

best_model_index = np.argmax(cv_scores)
best_model = models[best_model_index]
FINAL_THRESHOLD = mean_t

model_filename = f'lgbm_sber_base_seed_{RANDOM_SEED}_best_cv_model.pkl'
joblib.dump((best_model, FINAL_THRESHOLD), model_filename)
print(f"Лучшая модель и финальный порог ({FINAL_THRESHOLD:.3f}) сохранены.")

Разделение данных...
Вес положительного класса (scale_pos_weight): 0.38

Начало 5-кратной кросс-валидации...
--- ФОЛД 1/5 --- Оптимальный порог: 0.316, F1-Score: 0.8689
--- ФОЛД 2/5 --- Оптимальный порог: 0.333, F1-Score: 0.8608
--- ФОЛД 3/5 --- Оптимальный порог: 0.300, F1-Score: 0.8626
--- ФОЛД 4/5 --- Оптимальный порог: 0.300, F1-Score: 0.8698
--- ФОЛД 5/5 --- Оптимальный порог: 0.349, F1-Score: 0.8622
-------------------------------------------
Средний F1-Score по 5 фолдам: 0.8649 (±0.0037)
Средний Оптимальный Порог: 0.320
-------------------------------------------
Лучшая модель и финальный порог (0.320) сохранены.


In [4]:
import pandas as pd
import joblib

CUSTOMER_ID_COL = 'customer_id'


model_filename = f'lgbm_sber_base_seed_{RANDOM_SEED}_best_cv_model.pkl'
try:

    best_model, FINAL_THRESHOLD = joblib.load(model_filename)
    print(f"Загружена модель с финальным порогом: {FINAL_THRESHOLD:.3f}")
except FileNotFoundError:
    print(f"Критическая ошибка: Файл модели '{model_filename}' не найден.")
    raise SystemExit("Модель не сохранена.")

X_test = df_test_ready.copy()

test_proba = best_model.predict_proba(X_test)[:, 1]

test_predictions = (test_proba >= FINAL_THRESHOLD).astype(int)

df_submission = pd.DataFrame({
    CUSTOMER_ID_COL: test_ids, 
    'accepted': test_predictions
})

df_submission['accepted'] = df_submission['accepted'].astype(int)

submission_path = f'submission_sber_base_seed_{RANDOM_SEED}_TUNE.csv'
df_submission.to_csv(submission_path, index=False)

print(f"\nСоздан файл для отправки: {submission_path}")
print("Первые 10 строк файла:")
display(df_submission.head(10))

Загружена модель с финальным порогом: 0.320

Создан файл для отправки: submission_sber_base_seed_42_TUNE.csv
Первые 10 строк файла:


,customer_id,accepted
0,6001,0
1,6003,1
2,6005,1
3,6007,1
4,6009,1
5,6011,1
6,6013,1
7,6015,1
8,6017,1
9,6019,0
